# Building Image Generation Applications (OpenAI)

There's more to LLMs than text generation. You can also generate images from text descriptions. Images as a modality are useful across MedTech, architecture, tourism, game development, marketing, and more. In this lesson we work with today's **GPT Image** models on the OpenAI platform.

## Learning goals

By the end of this lesson you'll be able to:

- Explain what image generation is and where it's useful.
- Understand the `gpt-image` model family and how it differs from the legacy DALL·E models.
- Build an image generation application and edit images.

## What is image generation?

Image generation models create images from a text prompt. Modern models such as `gpt-image` learn the relationship between text and images during training, then iteratively turn random noise into an image that matches your prompt.

Two well-known families of image models are:

- **`gpt-image` (OpenAI)** — the current generation used in this lesson. It supports text-to-image generation and image editing (inpainting with a mask).
- **Midjourney** — a popular third-party model with its own service and Discord-based workflow.

> The older OpenAI image models — **DALL·E 2** and **DALL·E 3** — are legacy. DALL·E 3 is no longer available for new deployments, and features like `create_variation` only existed in DALL·E 2. Use the `gpt-image` models for new applications.

> **Important:** `gpt-image` models return the generated image as **base64** (`b64_json`), not as a URL. Your code decodes the base64 string to bytes and saves it — there's no image URL to download.


## 建立你的第一個圖像生成應用程式

那麼，建立一個圖像生成應用程式需要甚麼？你需要以下的函式庫：

- **python-dotenv**，強烈建議你使用此函式庫，將秘密資訊保存在 *.env* 檔案中，與程式碼分開。
- **openai**，此函式庫用於與 OpenAI API 互動。
- **pillow**，用於在 Python 中處理圖像。


1. 建立一個名為 *.env* 的檔案，內容如下：

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. 收集上述函式庫到一個名為 *requirements.txt* 的檔案，如下所示：

    ```text
    python-dotenv
    openai
    pillow
    ```


1. 接著，建立虛擬環境並安裝函式庫：


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> 對於 Windows，用以下指令來建立及啟動你的虛擬環境：

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. 在名為 *app.py* 的檔案中加入以下程式碼：

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # 建立 OpenAI 物件（從你的 .env 讀取 OPENAI_API_KEY）
    client = openai.OpenAI()


    try:
        # 使用圖像生成 API 建立圖像
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # 在此輸入你的提示文字
            size='1024x1024',
            n=1
        )
        # 設定儲存圖像的目錄
        image_dir = os.path.join(os.curdir, 'images')

        # 如果目錄不存在，則建立它
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # 初始化圖像路徑（注意檔案類型應該是 png）
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image 模型以 base64 (b64_json) 格式返回圖像，而非 URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # 在預設圖像檢視器中顯示圖像
        image = Image.open(image_path)
        image.show()

    # 捕捉異常
    except openai.BadRequestError as err:
        print(err)

    ```

讓我們來說明這段程式碼：

- 首先，我們匯入需要的函式庫，包括 OpenAI 函式庫、dotenv 函式庫、base64 模組，以及 Pillow 函式庫。

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- 接著，我們建立客戶端，會從你的 ``.env`` 檔案讀取 API 金鑰。

    ```python
    # 建立 OpenAI 物件
    client = openai.OpenAI()
    ```

- 然後，我們產生圖片：

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # 在此輸入您的提示文字
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` 型號會以 **base64** 字串格式回傳圖片於 `data[0].b64_json`，我們再將它解碼為位元組並寫入檔案 — 沒有可下載的 URL。

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- 最後，我們開啟圖片並使用標準圖片瀏覽器來顯示它：

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### 關於圖片產生的更多細節

讓我們來看看 `images.generate` 的參數：

- **model**，要使用的圖片模型（例如 `gpt-image-1`）。
- **prompt**，用來產生圖片的文字提示。這裡是「Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils」。
- **size**，產生圖片的大小（例如 `1024x1024`、`1536x1024`、`1024x1536` 或 `"auto"`）。
- **n**，產生圖片的數量。這裡我們產生一張。

> 影像模型不會接受 `temperature` 參數 — 那是文字生成的控制。若要取得多樣性，可再次呼叫 API；若要減少多樣性，請讓提示更具體。

## 圖片生成的其他功能

你已經看過如何用幾行 Python 產生圖片。`gpt-image` 模型也可以<strong>編輯</strong>現有圖片。透過提供一張圖片、可選的<strong>遮罩</strong>（標示出欲更改的範圍）以及提示，你可以更改圖片的部分區域 — 例如，幫我們的兔子加頂帽子。

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# 編輯同樣會以 base64 格式返回
edited_image = base64.b64decode(response.data[0].b64_json)
```

原圖只包含兔子；最終圖案上則有兔子戴著帽子。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
